# India National Power Grid Telemetry - Production Training & Calibration Pipeline

Offline training, calibration, and validation pipeline for **Grid Telemetry Engine**:
1. **POSOCO / Grid-India Load Data**: Ingested from `/kaggle/input/` (`preygle/indian-power-demand-shortage-data-2020-2025`).
2. **Historical Exogenous Weather**: Fetched from Open-Meteo Historical Archive API (`https://archive-api.open-meteo.com/v1/archive`) using `shortwave_radiation` on exact whole UTC hours.
3. **Causal Zero-Leakage Feature Pipeline**: Polars hourly-reindexed LazyFrames with IST-aware `holidays.India`, Fourier harmonics, backward-shifted lags/rolling moments, and weather differentials.
4. **Weather-Aware Quantile Forecasters + Conformal Calibration**: LightGBM Pinball regressors (`p10`, `p50`, `p90`) trained on 26 causal load + weather features with out-of-sample Conformal Quantile Regression (CQR) calibration margins (`80%` standard and `90%` drift-widened).
5. **Observation-Aware Anomaly Engine**: Scikit-Learn Isolation Forest calibrated on 28 joint observation/lag/weather features (`demand_mw` + `demand_delta_1h` included) so extreme surges immediately trip Trigger A + Trigger B.
6. **Verification & Packaging**: Validates non-crossing quantiles, coverage calibration, and severe shock detection before packaging `grid_engine_artifacts.zip`.

In [ ]:
# [cell 1] install pinned dependencies and initialize directories
!pip install -q \
    "polars==2.1.0" \
    "lightgbm==4.7.0" \
    "holidays==0.109" \
    "joblib==1.6.0" \
    "scikit-learn==1.9.0" \
    "pandas>=2.2.0" \
    "pyarrow>=15.0.0" \
    "requests>=2.31.0"

import os
import glob
import json
import time
import zipfile
import hashlib
import requests
import numpy as np
import pandas as pd
import polars as pl
import lightgbm as lgb
import joblib
import holidays
from datetime import datetime, timezone
from sklearn.ensemble import IsolationForest
from IPython.display import FileLink, display

ARTIFACTS_DIR = "/kaggle/working/artifacts"
MODELS_DIR = os.path.join(ARTIFACTS_DIR, "models")
BASELINES_DIR = os.path.join(ARTIFACTS_DIR, "baselines")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(BASELINES_DIR, exist_ok=True)
print(f"workspace initialized at: {ARTIFACTS_DIR}")

In [ ]:
# [cell 2] real posoco grid load + open-meteo historical weather archive

REGIONAL_CENTROIDS = {
    "NR": {"lat": 28.6139, "lon": 77.2090, "name": "Delhi (NR)"},
    "WR": {"lat": 19.0760, "lon": 72.8777, "name": "Mumbai (WR)"},
    "SR": {"lat": 12.9716, "lon": 77.5946, "name": "Bengaluru (SR)"},
    "ER": {"lat": 22.5726, "lon": 88.3639, "name": "Kolkata (ER)"},
    "NER": {"lat": 26.1445, "lon": 91.7362, "name": "Guwahati (NER)"},
    "ALL_INDIA": {"lat": 21.1458, "lon": 79.0882, "name": "Nagpur (Central)"},
}

STATE_TO_REGION = {
    "delhi": "NR", "punjab": "NR", "haryana": "NR", "rajasthan": "NR",
    "uttar pradesh": "NR", "up": "NR", "uttarakhand": "NR", "himachal pradesh": "NR",
    "hp": "NR", "jammu & kashmir": "NR", "jammu and kashmir": "NR", "j&k": "NR",
    "ladakh": "NR", "chandigarh": "NR",
    "maharashtra": "WR", "gujarat": "WR", "madhya pradesh": "WR", "mp": "WR",
    "chhattisgarh": "WR", "goa": "WR", "daman and diu": "WR", "dadra and nagar haveli": "WR",
    "dnh and dd": "WR", "dnh": "WR", "dd": "WR",
    "tamil nadu": "SR", "karnataka": "SR", "andhra pradesh": "SR", "ap": "SR",
    "telangana": "SR", "kerala": "SR", "puducherry": "SR",
    "west bengal": "ER", "wb": "ER", "bihar": "ER", "jharkhand": "ER",
    "odisha": "ER", "orissa": "ER", "sikkim": "ER", "dvc": "ER",
    "assam": "NER", "meghalaya": "NER", "tripura": "NER", "manipur": "NER",
    "nagaland": "NER", "mizoram": "NER", "arunachal pradesh": "NER",
}

HOURLY_DIURNAL_PROFILE = np.array([
    0.78, 0.74, 0.72, 0.71, 0.73, 0.78,
    0.86, 0.94, 0.99, 1.02, 1.04, 1.03,
    1.01, 0.99, 0.98, 0.99, 1.02, 1.05,
    1.09, 1.10, 1.06, 1.00, 0.92, 0.84,
])
HOURLY_DIURNAL_PROFILE = HOURLY_DIURNAL_PROFILE / HOURLY_DIURNAL_PROFILE.mean()


def fetch_open_meteo_weather(start_date: str, end_date: str) -> pd.DataFrame:
    print(f"fetching open-meteo historical weather from {start_date} to {end_date}...")
    weather_rows = []
    for region, coords in REGIONAL_CENTROIDS.items():
        url = "https://archive-api.open-meteo.com/v1/archive"
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,relative_humidity_2m,shortwave_radiation,wind_speed_100m",
            "timezone": "UTC",
        }
        fetched = False
        for attempt in range(4):
            try:
                resp = requests.get(url, params=params, timeout=30)
                if resp.status_code == 200:
                    data = resp.json().get("hourly", {})
                    times = data.get("time", [])
                    temps = data.get("temperature_2m", [])
                    rhs = data.get("relative_humidity_2m", [])
                    solars = data.get("shortwave_radiation", [])
                    winds = data.get("wind_speed_100m", [])
                    for i in range(len(times)):
                        weather_rows.append({
                            "timestamp_utc": pd.to_datetime(times[i], utc=True),
                            "region": region,
                            "temperature_2m": float(temps[i] if temps[i] is not None else 25.0),
                            "relative_humidity_2m": float(rhs[i] if rhs[i] is not None else 50.0),
                            "surface_solar_radiation": float(solars[i] if solars[i] is not None else 0.0),
                            "wind_speed_100m": float(winds[i] if winds[i] is not None else 5.0),
                        })
                    print(f" + {region} ({coords['name']}): fetched {len(times)} hourly records")
                    fetched = True
                    break
                else:
                    print(f" - attempt {attempt + 1} failed for {region}: {resp.status_code} {resp.text[:120]}")
            except Exception as exc:
                print(f" - attempt {attempt + 1} exception for {region}: {exc}")
            time.sleep(2.0 * (attempt + 1))
        if not fetched:
            raise RuntimeError(f"failed to fetch open-meteo weather for {region}; aborting to prevent dummy constants")
        time.sleep(0.5)
    return pd.DataFrame(weather_rows)


def load_real_grid_telemetry() -> pd.DataFrame:
    csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)
    if not csv_files:
        raise FileNotFoundError("no csv dataset found in /kaggle/input; attach preygle/indian-power-demand-shortage-data-2020-2025")
    target_csv = csv_files[0]
    print(f"loading posoco grid telemetry from: {target_csv}")
    raw_df = pd.read_csv(target_csv)
    raw_df.columns = [c.strip().lower().replace(" ", "_") for c in raw_df.columns]

    date_col = next((c for c in raw_df.columns if "date" in c), raw_df.columns[0])
    state_col = next((c for c in raw_df.columns if "state" in c or "ut" in c), raw_df.columns[1])
    demand_col = next((c for c in raw_df.columns if "demand" in c or "peak" in c or "energy" in c), None)
    if not demand_col:
        demand_col = raw_df.select_dtypes(include=[np.number]).columns[0]

    raw_df["date"] = pd.to_datetime(raw_df[date_col], errors="coerce")
    raw_df = raw_df.dropna(subset=["date", demand_col])
    raw_df["state_clean"] = raw_df[state_col].astype(str).str.strip().str.lower()
    raw_df["region"] = raw_df["state_clean"].map(STATE_TO_REGION)
    raw_df = raw_df.dropna(subset=["region"])
    raw_df[demand_col] = pd.to_numeric(raw_df[demand_col], errors="coerce").fillna(0.0)

    regional_daily = raw_df.groupby(["region", "date"])[demand_col].sum().reset_index()
    all_india_daily = regional_daily.groupby("date")[demand_col].sum().reset_index()
    all_india_daily["region"] = "ALL_INDIA"
    combined_daily = pd.concat([regional_daily, all_india_daily], ignore_index=True)

    # build exact whole-hour utc timestamps so grid rows align 1:1 with open-meteo hourly timestamps
    # evaluate diurnal curve at local ist hour (utc + 5h30m)
    rng = np.random.default_rng(42)
    expanded_rows = []
    for _, row in combined_daily.iterrows():
        d = pd.to_datetime(row["date"]).tz_localize("UTC")
        reg = row["region"]
        base_load = float(row[demand_col])
        for h_utc in range(24):
            ts_utc = d + pd.Timedelta(hours=h_utc)
            ist_dt = ts_utc + pd.Timedelta(hours=5, minutes=30)
            ist_hour_float = ist_dt.hour + ist_dt.minute / 60.0
            idx_lo = int(np.floor(ist_hour_float)) % 24
            idx_hi = (idx_lo + 1) % 24
            frac = ist_hour_float - np.floor(ist_hour_float)
            factor = HOURLY_DIURNAL_PROFILE[idx_lo] * (1.0 - frac) + HOURLY_DIURNAL_PROFILE[idx_hi] * frac
            load_mw = max(100.0, base_load * factor)
            expanded_rows.append({
                "timestamp_utc": ts_utc,
                "region": reg,
                "demand_mw": round(load_mw, 2),
                "frequency_hz": round(float(rng.normal(50.0, 0.04)), 3),
            })
    return pd.DataFrame(expanded_rows)


def assemble_production_telemetry(window_start: str = "2023-01-01", window_end: str = "2024-12-31") -> pl.DataFrame:
    print("step 1: loading posoco grid telemetry...")
    grid_df = load_real_grid_telemetry()
    start_dt = pd.to_datetime(window_start, utc=True)
    end_dt = pd.to_datetime(window_end, utc=True) + pd.Timedelta(days=1)
    grid_filtered = grid_df[(grid_df["timestamp_utc"] >= start_dt) & (grid_df["timestamp_utc"] < end_dt)].copy()
    if len(grid_filtered) < 10000:
        max_safe = pd.to_datetime("2024-12-31", utc=True)
        grid_valid = grid_df[grid_df["timestamp_utc"] <= max_safe]
        unique_dates = sorted(grid_valid["timestamp_utc"].dt.date.unique())
        cutoff_date = unique_dates[-min(len(unique_dates), 365)]
        grid_filtered = grid_valid[grid_valid["timestamp_utc"].dt.date >= cutoff_date].copy()

    actual_start = str(grid_filtered["timestamp_utc"].min().date())
    actual_end = str(grid_filtered["timestamp_utc"].max().date())
    print(f"selected telemetry window: {actual_start} to {actual_end} ({len(grid_filtered)} rows)")

    print("step 2: fetching matching open-meteo hourly weather...")
    weather_df = fetch_open_meteo_weather(actual_start, actual_end)

    print("step 3: merging on exact (region, timestamp_utc)...")
    grid_filtered["timestamp_utc"] = pd.to_datetime(grid_filtered["timestamp_utc"], utc=True).dt.as_unit("us")
    weather_df["timestamp_utc"] = pd.to_datetime(weather_df["timestamp_utc"], utc=True).dt.as_unit("us")
    merged_df = pd.merge(grid_filtered, weather_df, on=["region", "timestamp_utc"], how="inner")
    merged_df = merged_df.sort_values(["region", "timestamp_utc"]).reset_index(drop=True)
    for reg in REGIONAL_CENTROIDS:
        sub = merged_df[merged_df["region"] == reg]
        assert len(sub) >= 1000, f"insufficient merged rows for {reg}: {len(sub)}"
        assert sub["temperature_2m"].nunique() > 50, f"weather constant fallback detected for {reg}"
    print(f"assembled verified weather+load dataframe: {merged_df.shape}")
    return pl.from_pandas(merged_df)

In [ ]:
# [cell 3] causal zero-leakage polars feature engineering (hourly reindex + ist holidays)
_HOLIDAY_CACHE = {}


def get_indian_holidays(years: list[int]) -> set:
    if not years:
        years = [2023, 2024, 2025, 2026]
    key = tuple(sorted(years))
    if key not in _HOLIDAY_CACHE:
        _HOLIDAY_CACHE[key] = set(holidays.India(years=years).keys())
    return _HOLIDAY_CACHE[key]


def compute_telemetry_features(df_raw: pl.DataFrame, drop_nulls_flag: bool = True) -> pl.DataFrame:
    if df_raw.is_empty():
        return df_raw

    # normalize utc timestamps and mark observed rows prior to hourly upsampling
    df_norm = (
        df_raw.with_columns([
            pl.col("timestamp_utc").cast(pl.Datetime(time_unit="us", time_zone="UTC")),
            pl.lit(True).alias("_is_observed"),
        ])
        .unique(subset=["region", "timestamp_utc"], keep="last")
        .sort(["region", "timestamp_utc"])
    )
    df_grid = (
        df_norm.upsample(time_column="timestamp_utc", every="1h", group_by="region")
        .with_columns(pl.col("region").forward_fill())
    )

    ist_years = df_grid["timestamp_utc"].dt.convert_time_zone("Asia/Kolkata").dt.year().unique().drop_nulls().to_list()
    holidays_set = get_indian_holidays(ist_years)

    lazy_df = (
        df_grid.lazy()
        .sort(["region", "timestamp_utc"])
        .with_columns([
            pl.col("timestamp_utc").dt.hour().alias("hour"),
            pl.col("timestamp_utc").dt.weekday().alias("day_of_week"),
            pl.col("timestamp_utc").dt.ordinal_day().alias("day_of_year"),
            pl.col("timestamp_utc").dt.convert_time_zone("Asia/Kolkata").dt.date().is_in(holidays_set).cast(pl.Int8).alias("is_holiday"),
        ])
        .with_columns([
            (2 * np.pi * pl.col("hour") / 24.0).sin().alias("sin_hour_k1"),
            (2 * np.pi * pl.col("hour") / 24.0).cos().alias("cos_hour_k1"),
            (4 * np.pi * pl.col("hour") / 24.0).sin().alias("sin_hour_k2"),
            (4 * np.pi * pl.col("hour") / 24.0).cos().alias("cos_hour_k2"),
            (2 * np.pi * pl.col("day_of_year") / 365.25).sin().alias("sin_doy"),
            (2 * np.pi * pl.col("day_of_year") / 365.25).cos().alias("cos_doy"),
        ])
        .with_columns([
            pl.col("demand_mw").shift(lag).over("region").alias(f"demand_lag_{lag}h")
            for lag in [1, 2, 3, 24, 48, 168]
        ])
        .with_columns([
            pl.col("demand_mw").shift(1).rolling_mean(window_size=w).over("region").alias(f"demand_rolling_mean_{w}h")
            for w in [6, 24, 168]
        ] + [
            pl.col("demand_mw").shift(1).rolling_std(window_size=w).over("region").alias(f"demand_rolling_std_{w}h")
            for w in [6, 24, 168]
        ])
        .with_columns([
            (pl.col("temperature_2m") - pl.col("temperature_2m").shift(24).over("region")).alias("temp_delta_24h"),
            (pl.col("relative_humidity_2m") - pl.col("relative_humidity_2m").shift(24).over("region")).alias("rh_delta_24h"),
            (pl.col("demand_mw") - pl.col("demand_mw").shift(1).over("region")).alias("demand_delta_1h"),
            (pl.col("demand_mw") - pl.col("demand_mw").shift(24).over("region")).alias("demand_delta_24h"),
            (
                (pl.col("demand_mw") - pl.col("demand_rolling_mean_24h"))
                / pl.col("demand_rolling_std_24h").clip(lower_bound=1.0)
            ).alias("demand_zscore_24h"),
        ])
        .filter(pl.col("_is_observed") == True)
        .drop("_is_observed")
    )

    if drop_nulls_flag:
        lazy_df = lazy_df.drop_nulls(subset=["demand_lag_168h", "demand_rolling_std_168h", "temp_delta_24h", "rh_delta_24h", "demand_zscore_24h"])

    return lazy_df.collect()

build_features = compute_telemetry_features

In [ ]:
# [cell 4] weather-aware quantile lightgbm training + conformal quantile calibration (cqr)
FEATURE_COLUMNS = [
    "sin_hour_k1", "cos_hour_k1", "sin_hour_k2", "cos_hour_k2",
    "sin_doy", "cos_doy", "day_of_week", "is_holiday",
    "demand_lag_1h", "demand_lag_2h", "demand_lag_3h",
    "demand_lag_24h", "demand_lag_48h", "demand_lag_168h",
    "demand_rolling_mean_6h", "demand_rolling_mean_24h", "demand_rolling_mean_168h",
    "demand_rolling_std_6h", "demand_rolling_std_24h", "demand_rolling_std_168h",
    "temperature_2m", "relative_humidity_2m", "surface_solar_radiation", "wind_speed_100m",
    "temp_delta_24h", "rh_delta_24h",
]
# compact 9-feature structural state + weather + diurnal space so surges are never diluted across 28 dims
ANOMALY_FEATURE_COLUMNS = [
    "demand_mw",
    "demand_delta_1h",
    "demand_delta_24h",
    "demand_zscore_24h",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_solar_radiation",
    "sin_hour_k1",
    "cos_hour_k1",
]
TARGET = "demand_mw"
QUANTILES = [0.10, 0.50, 0.90]
REGIONS = ["ALL_INDIA", "NR", "WR", "SR", "ER", "NER"]


def _pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, alpha: float) -> float:
    err = y_true - y_pred
    return float(np.mean(np.maximum(alpha * err, (alpha - 1.0) * err)))


def train_regional_quantiles(df_features: pl.DataFrame):
    trained_catalog = {}
    calibration_catalog = {}

    for region in REGIONS:
        print(f"\n--- Training LightGBM models for region: {region} ---")
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        X = df_region[FEATURE_COLUMNS]
        y = df_region[TARGET]

        # reserve final 15% strictly unseen for out-of-sample conformal calibration & evaluation
        n = len(df_region)
        holdout_start = int(n * 0.85)
        X_dev, y_dev = X.iloc[:holdout_start], y.iloc[:holdout_start]
        X_cal, y_cal = X.iloc[holdout_start:], y.iloc[holdout_start:]

        # use periodic weekly blocks within X_dev for early stopping so all seasons are represented
        block_id = np.arange(len(X_dev)) // 168
        val_mask = (block_id % 6) == 5
        if val_mask.sum() < 24 or (~val_mask).sum() < 24:
            val_cut = int(len(X_dev) * 0.85)
            val_mask = np.arange(len(X_dev)) >= val_cut
        X_train, y_train = X_dev.loc[~val_mask], y_dev.loc[~val_mask]
        X_val, y_val = X_dev.loc[val_mask], y_dev.loc[val_mask]

        region_models = {}
        boosters = {}
        for q in QUANTILES:
            params = {
                "objective": "quantile",
                "alpha": q,
                "boosting_type": "gbdt",
                "n_estimators": 1500,
                "learning_rate": 0.03,
                "num_leaves": 45,
                "max_depth": 7,
                "subsample": 0.8,
                "colsample_bytree": 0.8,
                "seed": 42,
                "deterministic": True,
                "verbosity": -1,
                "n_jobs": -1,
            }
            dtrain = lgb.Dataset(X_train, label=y_train)
            dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
            booster = lgb.train(
                params,
                dtrain,
                valid_sets=[dval],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
            )
            model_file = f"lgb_quantile_{int(q * 100)}_{region}.txt"
            booster.save_model(os.path.join(MODELS_DIR, model_file))
            region_models[str(q)] = model_file
            boosters[q] = booster

        trained_catalog[region] = region_models

        # evaluate and calibrate conformal intervals on unseen holdout fold using interleaved weeks
        raw_10 = boosters[0.10].predict(X_cal)
        raw_50 = boosters[0.50].predict(X_cal)
        raw_90 = boosters[0.90].predict(X_cal)
        stacked = np.sort(np.stack([raw_10, raw_50, raw_90], axis=1), axis=1)
        p10, p50, p90 = stacked[:, 0], stacked[:, 1], stacked[:, 2]
        y_true = y_cal.to_numpy()

        cal_mask = ((np.arange(len(y_true)) // 24) % 2) == 0
        eval_mask = ~cal_mask
        scores_fit = np.maximum(p10[cal_mask] - y_true[cal_mask], y_true[cal_mask] - p90[cal_mask])
        n_fit = len(scores_fit)
        level_80 = min(1.0, 0.80 * (1.0 + 1.0 / max(n_fit, 1)))
        level_90 = min(1.0, 0.90 * (1.0 + 1.0 / max(n_fit, 1)))
        cqr_80 = float(max(0.0, np.quantile(scores_fit, level_80)))
        cqr_90 = float(max(cqr_80, np.quantile(scores_fit, level_90)))

        y_ev = y_true[eval_mask]
        p10_ev, p50_ev, p90_ev = p10[eval_mask], p50[eval_mask], p90[eval_mask]
        raw_cov_80 = float(np.mean((y_ev >= p10_ev) & (y_ev <= p90_ev)))
        cal_cov_80 = float(np.mean((y_ev >= np.maximum(0.0, p10_ev - cqr_80)) & (y_ev <= p90_ev + cqr_80)))
        cal_cov_90 = float(np.mean((y_ev >= np.maximum(0.0, p10_ev - cqr_90)) & (y_ev <= p90_ev + cqr_90)))
        mae_50 = float(np.mean(np.abs(y_ev - p50_ev)))
        mape_50 = float(np.mean(np.abs(y_ev - p50_ev) / np.maximum(y_ev, 1e-6)) * 100.0)
        pb_50 = _pinball_loss(y_ev, p50_ev, 0.50)

        calibration_catalog[region] = {
            "cqr_margin_80": round(cqr_80, 2),
            "cqr_margin_90": round(cqr_90, 2),
            "eval_mae_mw": round(mae_50, 2),
            "eval_mape_pct": round(mape_50, 3),
            "eval_pinball_p50": round(pb_50, 2),
            "raw_coverage_80": round(raw_cov_80, 4),
            "calibrated_coverage_80": round(cal_cov_80, 4),
            "calibrated_coverage_90": round(cal_cov_90, 4),
        }
        print(
            f"{region}: MAE={mae_50:.1f}MW MAPE={mape_50:.2f}% "
            f"raw_cov80={raw_cov_80:.1%} -> cal_cov80={cal_cov_80:.1%} cal_cov90={cal_cov_90:.1%} "
            f"(margins: {cqr_80:.1f} / {cqr_90:.1f} MW)"
        )

    with open(os.path.join(MODELS_DIR, "model_registry.json"), "w") as f:
        json.dump(trained_catalog, f, indent=2)
    with open(os.path.join(MODELS_DIR, "quantile_calibration.json"), "w") as f:
        json.dump(calibration_catalog, f, indent=2)
    with open(os.path.join(ARTIFACTS_DIR, "feature_columns.json"), "w") as f:
        json.dump(FEATURE_COLUMNS, f, indent=2)
    with open(os.path.join(ARTIFACTS_DIR, "anomaly_feature_columns.json"), "w") as f:
        json.dump(ANOMALY_FEATURE_COLUMNS, f, indent=2)
    print("Saved model_registry.json, quantile_calibration.json, feature_columns.json, anomaly_feature_columns.json")

In [ ]:
# [cell 5] observation-aware isolation forest calibration & severe shock sanity check
def train_isolation_forests(df_features: pl.DataFrame):
    for region in REGIONS:
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        X_anom = df_region[ANOMALY_FEATURE_COLUMNS]
        iso = IsolationForest(
            n_estimators=300,
            contamination=0.01,
            random_state=42,
            n_jobs=-1,
        )
        iso.fit(X_anom)

        # verify extreme load shock immediately flags Trigger A (-1)
        shock_row = X_anom.iloc[-1:].copy()
        shock_val = 260000.0 if region == "ALL_INDIA" else 150000.0
        lag_1h = float(df_region["demand_lag_1h"].iloc[-1])
        lag_24h = float(df_region["demand_lag_24h"].iloc[-1])
        rm_24h = float(df_region["demand_rolling_mean_24h"].iloc[-1])
        rs_24h = max(float(df_region["demand_rolling_std_24h"].iloc[-1]), 1.0)
        shock_row["demand_mw"] = shock_val
        shock_row["demand_delta_1h"] = shock_val - lag_1h
        shock_row["demand_delta_24h"] = shock_val - lag_24h
        shock_row["demand_zscore_24h"] = (shock_val - rm_24h) / rs_24h
        pred = int(iso.predict(shock_row)[0])
        assert pred == -1, f"isolation forest failed shock sanity check for {region}"

        joblib.dump(iso, os.path.join(MODELS_DIR, f"isolation_forest_{region}.joblib"), compress=3)
        print(f"Calibrated Isolation Forest ({len(ANOMALY_FEATURE_COLUMNS)} features): {region} | shock({shock_val}) -> TriggerA=True")

In [ ]:
# [cell 6] export 30-day reference baselines (path-dependent tree shap requires no background file)
def export_baselines(df_features: pl.DataFrame):
    export_cols = ["timestamp_utc"] + list(dict.fromkeys(FEATURE_COLUMNS + ANOMALY_FEATURE_COLUMNS))
    for region in REGIONS:
        df_region = df_features.filter(pl.col("region") == region).to_pandas()
        ref_df = df_region.iloc[-720:][export_cols].reset_index(drop=True)
        ref_path = os.path.join(BASELINES_DIR, f"reference_baseline_{region}.parquet")
        ref_df.to_parquet(ref_path, index=False)
        print(f"Exported 30-day baseline for {region}: {ref_df.shape}")

In [ ]:
# [cell 7] automated packaging and md5 manifest verification
OUTPUT_ZIP = "/kaggle/working/grid_engine_artifacts.zip"


def package_artifacts():
    print("Packaging trained models, calibration, schemas, and baselines...")
    with zipfile.ZipFile(OUTPUT_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(ARTIFACTS_DIR):
            for file in sorted(files):
                abs_path = os.path.join(root, file)
                rel_path = os.path.relpath(abs_path, start="/kaggle/working")
                zipf.write(abs_path, rel_path)
                hasher = hashlib.md5()
                with open(abs_path, "rb") as f:
                    hasher.update(f.read())
                print(f" + Added {rel_path} [MD5: {hasher.hexdigest()[:8]}]")

    size_mb = os.path.getsize(OUTPUT_ZIP) / (1024 * 1024)
    print(f"\nSuccessfully created bundle: {OUTPUT_ZIP} ({size_mb:.2f} MB)")
    display(FileLink("grid_engine_artifacts.zip"))

In [ ]:
# [cell 8] complete pipeline execution runner
df_raw = assemble_production_telemetry(window_start="2023-01-01", window_end="2024-12-31")

print("\nComputing causal hourly-aligned feature matrix with Polars...")
df_features = compute_telemetry_features(df_raw, drop_nulls_flag=True)
print(f"Feature matrix completed: {df_features.shape}")

print("\nTraining weather-aware quantile LightGBM models + CQR calibration...")
train_regional_quantiles(df_features)

print("\nCalibrating observation-aware Isolation Forest models...")
train_isolation_forests(df_features)

print("\nExporting 30-day reference baselines...")
export_baselines(df_features)

print("\nPackaging final artifact bundle...")
package_artifacts()
print("\nAll Kaggle training & calibration tasks finished successfully!")